# 02 — Mushroom: definitieve data preparation

Dit notebook zet de oorspronkelijke Secondary Mushroom CSV om naar voorbereide data. Het herhaalt alleen de gekozen stappen uit `01_clean_en_explore.ipynb`, zonder grafieken of modeltraining. Je kan dit notebook zelfstandig van boven naar beneden uitvoeren.

**Wie werkte mee?**

- Student(en): vul hier jullie namen en werkelijk gecontroleerde/uitgewerkte bijdragen in.
- AI-hulp: Codex hielp bij de code, structuur, Nederlandse uitleg en controle. De student(en) controleren en verdedigen de keuzes.

**Basis uit de lessen:** consistente kolomnamen uit Tidy data, ontbrekende waarden beoordelen uit Titanic, `CategoricalDtype` uit Diamonds en `pd.get_dummies()` uit MPG. De vergelijking van de cleaningkeuzes staat in notebook 01.

We behouden 60.923 unieke voorbeelden en alle 20 invoerkenmerken. We schatten geen ontbrekende metingen of categorieën, verwijderen geen statistische outliers en schalen nog niet.

## 1. Libraries en oorspronkelijke data inladen

We gebruiken dezelfde CSV als de EDA. Het scheidingsteken is een puntkomma. Het oorspronkelijke bestand wordt niet overschreven. Selecteer een kernel met Pandas, NumPy en IPython, bijvoorbeeld de bestaande `.venv-1`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.api.types import CategoricalDtype
from IPython.display import display

In [2]:
# We zoeken de CSV vanuit de projectmap of vanuit de map van dit notebook.
# Zo hoef je geen pad met de naam van jouw computer in de code te zetten.
werkmap = Path.cwd()
if (werkmap / 'MushroomDataset' / 'MushroomDataset' / 'secondary_data.csv').exists():
    mushroom_map = werkmap / 'MushroomDataset'
else:
    mushroom_map = werkmap

data_pad = mushroom_map / 'MushroomDataset' / 'secondary_data.csv'
if not data_pad.exists():
    raise FileNotFoundError('Open dit notebook vanuit CloudAI of vanuit CloudAI/MushroomDataset.')

# In dit bestand worden de kolommen gescheiden door puntkomma's.
df_raw = pd.read_csv(data_pad, sep=';')
print(f'Aantal rijen: {df_raw.shape[0]:,}')
print(f'Aantal kolommen: {df_raw.shape[1]}')
display(df_raw.head())

Aantal rijen: 61,069
Aantal kolommen: 21


,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,p,15.26,x,g,o,f,e,NaN,w,16.95,...,s,y,w,u,w,t,g,NaN,d,w
1,p,16.60,x,g,o,f,e,NaN,w,17.99,...,s,y,w,u,w,t,g,NaN,d,u
2,p,14.07,x,g,o,f,e,NaN,w,17.80,...,s,y,w,u,w,t,g,NaN,d,w
3,p,14.17,f,h,e,f,e,NaN,w,15.77,...,s,y,w,u,w,t,p,NaN,d,w
4,p,14.64,x,h,o,f,e,NaN,w,16.53,...,s,y,w,u,w,t,p,NaN,d,w


## 2. Consistente kolomnamen en ontbrekende codes

We maken de namen gelijkvormig en onderscheiden numerieke kenmerken, categorische kenmerken en de doelklasse. We behandelen lege tekst en een eventuele `?` als onbekend. Een code zoals `f` laten we staan.

In [3]:
# We bewaren de originele data en werken verder met een kopie.
df = df_raw.copy()

# Zoals in de tidy-dataoefening gebruiken we consistente kolomnamen.
df.columns = df.columns.str.strip().str.lower().str.replace('-', '_', regex=False)

# Dit zijn de drie echte metingen. De breedte staat in mm, de andere twee in cm.
numerieke_kolommen = ['cap_diameter', 'stem_height', 'stem_width']
categorische_kolommen = [kolom for kolom in df.columns
                        if kolom not in numerieke_kolommen + ['class']]

# We verwijderen alleen extra spaties, niet de betekenisvolle lettercodes.
for kolom in categorische_kolommen + ['class']:
    df[kolom] = df[kolom].str.strip()
    df[kolom] = df[kolom].replace({'': np.nan, '?': np.nan})

display(pd.DataFrame({'originele_naam': df_raw.columns, 'nieuwe_naam': df.columns}))

,originele_naam,nieuwe_naam
0,class,class
1,cap-diameter,cap_diameter
2,cap-shape,cap_shape
3,cap-surface,cap_surface
4,cap-color,cap_color
5,does-bruise-or-bleed,does_bruise_or_bleed
6,gill-attachment,gill_attachment
7,gill-spacing,gill_spacing
8,gill-color,gill_color
9,stem-height,stem_height


## 3. Categorieën en doelklasse controleren

De lijst komt uit het lokale metadata-bestand. De CSV heeft daarnaast `d` bij hoedoppervlak en `f` bij steelbasis. We bewaren die extra broncodes zonder hun betekenis te raden.

Een nieuwe, niet-herkende code doet de uitvoering stoppen. Zo kunnen we die eerst onderzoeken in plaats van ze ongemerkt als leeg te bewaren.

In [4]:
# Deze mogelijke codes komen uit secondary_data_meta.txt.
# 'd' bij cap_surface en 'f' bij stem_root staan wel in de CSV, maar niet in die uitleg.
# We bewaren ze als aparte codes. We verzinnen er geen beschrijving bij.
kleuren = ['n', 'b', 'g', 'r', 'p', 'u', 'e', 'w', 'y', 'l', 'o', 'k']
oppervlakken = ['i', 'g', 'y', 's', 'h', 'l', 'k', 't', 'w', 'e']
mogelijke_codes = {
    'cap_shape': ['b', 'c', 'x', 'f', 's', 'p', 'o'],
    'cap_surface': oppervlakken + ['d'],
    'cap_color': kleuren,
    'does_bruise_or_bleed': ['t', 'f'],
    'gill_attachment': ['a', 'x', 'd', 'e', 's', 'p', 'f'],
    'gill_spacing': ['c', 'd', 'f'],
    'gill_color': kleuren + ['f'],
    'stem_root': ['b', 's', 'c', 'u', 'e', 'z', 'r', 'f'],
    'stem_surface': oppervlakken + ['f'],
    'stem_color': kleuren + ['f'],
    'veil_type': ['p', 'u'],
    'veil_color': kleuren + ['f'],
    'has_ring': ['t', 'f'],
    'ring_type': ['c', 'e', 'r', 'g', 'l', 'p', 's', 'z', 'y', 'm', 'f'],
    'spore_print_color': kleuren,
    'habitat': ['g', 'l', 'm', 'p', 'h', 'u', 'w', 'd'],
    'season': ['s', 'u', 'a', 'w']
}

# We controleren de waarden VOOR we er categoricals van maken.
# Anders zou een niet-herkende code ongemerkt een ontbrekende waarde kunnen worden.
for kolom, codes in mogelijke_codes.items():
    extra_codes = set(df[kolom].dropna().unique()) - set(codes)
    if extra_codes:
        raise ValueError(f'Controleer nieuwe codes in {kolom}: {extra_codes}')

if df['class'].isna().any() or not set(df['class'].unique()).issubset({'e', 'p'}):
    raise ValueError('De doelkolom moet voor iedere rij e of p bevatten.')

print('Alle categorieën zijn gecontroleerd. De twee extra broncodes blijven behouden.')

Alle categorieën zijn gecontroleerd. De twee extra broncodes blijven behouden.


## 4. Alleen volledige herhalingen verwijderen

Uit notebook 01 blijkt dat dezelfde invoer geen verschillende doelklassen heeft. Dat controleren we hier opnieuw. Daarna verwijderen we alleen exacte herhalingen, inclusief dezelfde doelklasse. Alle 146 herhalingen zitten in klasse `p`; het aandeel van die klasse verandert maar weinig.

De oorspronkelijke metingen van de behouden rijen blijven gelijk. Een nulsteel of een grote meting blijft dus bewaard.

In [5]:
# Als een nieuwe bron tegenstrijdige labels heeft, moeten we die eerst onderzoeken.
invoerkolommen = numerieke_kolommen + categorische_kolommen
labels_per_invoer = df.groupby(invoerkolommen, dropna=False)['class'].nunique()
if (labels_per_invoer > 1).any():
    raise ValueError('Dezelfde invoer heeft verschillende doelklassen. Onderzoek dit eerst in de EDA.')

In [6]:
# We verwijderen alleen volledige herhalingen, inclusief dezelfde doelklasse.
# We maken geen keuze tussen verschillende labels voor dezelfde kenmerken.
df_clean = df.drop_duplicates().copy()
print(f'Rijen voor verwijderen: {len(df):,}')
print(f'Rijen na verwijderen: {len(df_clean):,}')
print(f'Verwijderde herhalingen: {len(df) - len(df_clean)}')

Rijen voor verwijderen: 61,069
Rijen na verwijderen: 60,923
Verwijderde herhalingen: 146


## 5. Onbekende informatie bewaren en categoricals vastleggen

We maken een aparte categorie `onbekend` voor lege categorische cellen. We raden geen ringtype, kleur of vorm. De bestaande codes voor 'geen' blijven apart staan.

Zoals in de Diamonds-oefening gebruiken we `CategoricalDtype`. Hier zijn alle invoercategorieën nominaal, dus `ordered=False`. Seizoenen krijgen geen kunstmatige oplopende score.

In [7]:
# Een ontbrekende waarde wordt 'onbekend'. Dat is geen voorspelde kleur of vorm.
# De bestaande code 'f' blijft bestaan. De betekenis van die code hangt van de kolom af.
for kolom in categorische_kolommen:
    df_clean[kolom] = df_clean[kolom].fillna('onbekend')

    # Zoals in de Diamonds-oefening gebruiken we CategoricalDtype.
    # Deze categorieën hebben geen grootte of rangorde, dus ordered=False.
    categorie_type = CategoricalDtype(
        categories=mogelijke_codes[kolom] + ['onbekend'], ordered=False
    )
    df_clean[kolom] = df_clean[kolom].astype(categorie_type)

df_clean['class'] = df_clean['class'].astype(
    CategoricalDtype(categories=['e', 'p'], ordered=False)
)
print('Ontbrekende informatie is nu herkenbaar als de categorie onbekend.')

Ontbrekende informatie is nu herkenbaar als de categorie onbekend.


## 6. Numerieke invoer maken met one-hot encoding

`pd.get_dummies()` maakt een 0/1-kolom per mogelijke categorie, zoals in de MPG-oefening. De doelklasse wordt `0 = eetbaar` en `1 = giftig/onbekende eetbaarheid`.

De vaste categorieën zorgen voor dezelfde kolommen bij dezelfde bron. Sommige gedocumenteerde categorieën komen nu niet voor en leveren volledig nulkolommen op. We laten die staan voor het vaste schema; we mogen daaruit niet afleiden dat een model die ongeziene categorie al kan voorspellen.

We behouden cm en mm. Als een later model scaling of verdere selectie nodig heeft, bepalen we die parameters alleen op de trainingsset en passen we ze gelijk toe op validatie, test en nieuwe invoer. Dit notebook leert geen modus, gemiddelde, outliergrens of schaal uit de testset.

In [8]:
# One-hot encoding maakt voor iedere categorie een aparte 0/1-kolom.
# Zo betekent code 'w' bijvoorbeeld niet dat deze groter is dan code 'n'.
# De vaste categorieën hierboven bepalen de kolommen, niet een geleerd gemiddelde.
df_prepared = pd.get_dummies(df_clean, columns=categorische_kolommen, dtype=int)

# Alleen de doelklasse zetten we om: 0 = eetbaar, 1 = giftig/onbekende eetbaarheid.
df_prepared['class'] = df_clean['class'].map({'e': 0, 'p': 1}).astype(int)
print(f'Aantal invoerkenmerken voor encoding: {len(df_clean.columns) - 1}')
print(f'Aantal invoerkenmerken na encoding: {len(df_prepared.columns) - 1}')
display(df_prepared.iloc[:5, :10])

Aantal invoerkenmerken voor encoding: 20
Aantal invoerkenmerken na encoding: 159


,class,cap_diameter,stem_height,stem_width,cap_shape_b,cap_shape_c,cap_shape_x,cap_shape_f,cap_shape_s,cap_shape_p
0,1,15.26,16.95,17.09,0,0,1,0,0,0
1,1,16.60,17.99,18.19,0,0,1,0,0,0
2,1,14.07,17.80,17.74,0,0,1,0,0,0
3,1,14.17,15.77,15.98,0,0,0,1,0,0
4,1,14.64,16.53,17.20,0,0,1,0,0,0


## 7. Controleren vóór export

We controleren rijenverlies, de verdeling van de klassen, ontbrekende informatie, metingen en de encoding. Deze controles tonen dat de gekozen voorbereiding werkt. Ze zeggen nog niets over de score van een toekomstig model.

In [9]:
# De metingen moeten exact gelijk blijven aan de behouden oorspronkelijke rijen.
origineel_uniek = df.loc[df_clean.index]
assert df_clean[numerieke_kolommen].equals(origineel_uniek[numerieke_kolommen])
assert len(df_clean) == len(df) - int(df.duplicated().sum())
assert not df_clean.duplicated().any()
assert not df_clean.isna().any().any()
assert not df_prepared.isna().any().any()
assert df_prepared.select_dtypes(exclude='number').empty
assert set(df_prepared['class'].unique()) == {0, 1}

# Elke oude lege categorie moet nu expliciet onbekend zijn.
for kolom in categorische_kolommen:
    assert int((df_clean[kolom] == 'onbekend').sum()) == int(origineel_uniek[kolom].isna().sum())

# In ieder blok one-hot-kolommen moet iedere rij precies één 1 hebben.
for kolom in categorische_kolommen:
    dummy_kolommen = [f'{kolom}_{categorie}' for categorie in df_clean[kolom].cat.categories]
    assert (df_prepared[dummy_kolommen].sum(axis=1) == 1).all()

voor = df['class'].value_counts().reindex(['e', 'p'])
na = df_clean['class'].value_counts().reindex(['e', 'p'])
display(pd.DataFrame({
    'voor': voor,
    'na': na,
    'aandeel_voor_pct': (voor / len(df) * 100).round(3),
    'aandeel_na_pct': (na / len(df_clean) * 100).round(3)
}))
print('Alle controles zijn geslaagd.')

,voor,na,aandeel_voor_pct,aandeel_na_pct
class,,,,
e,27181,27181,44.509,44.615
p,33888,33742,55.491,55.385


Alle controles zijn geslaagd.


## 8. Voorbereide bestanden opslaan

We schrijven twee bestanden in `MushroomDataset/prepared/`:

- `mushroom_clean.csv`: de leesbare versie met lettercodes, expliciet `onbekend` en doelklasse `e/p`. Een CSV bewaart de Pandas-categoricaltypes niet; die leg je bij opnieuw inladen weer vast indien nodig.
- `mushroom_prepared.csv`: de numerieke versie met one-hot-kolommen en doelklasse `0/1`. Deze kan later als vertrekpunt voor numerieke modellen dienen.

We gebruiken `index=False`, zodat geen extra indexkolom wordt toegevoegd. Opnieuw uitvoeren maakt dezelfde exports opnieuw aan. De bronbestanden blijven bewaard. We laden de exports ook opnieuw in om te controleren dat het opslaan gelukt is.

In [10]:
# Deze map bevat alleen de afgeleide bestanden van dit notebook.
export_map = mushroom_map / 'prepared'
export_map.mkdir(exist_ok=True)
clean_pad = export_map / 'mushroom_clean.csv'
prepared_pad = export_map / 'mushroom_prepared.csv'

df_clean.to_csv(clean_pad, sep=';', index=False)
df_prepared.to_csv(prepared_pad, sep=';', index=False)

# We controleren ook de bestanden op schijf, niet alleen de tabellen in het geheugen.
clean_op_schijf = pd.read_csv(clean_pad, sep=';')
prepared_op_schijf = pd.read_csv(prepared_pad, sep=';')
assert clean_op_schijf.shape == df_clean.shape
assert prepared_op_schijf.shape == df_prepared.shape
assert clean_op_schijf.columns.tolist() == df_clean.columns.tolist()
assert prepared_op_schijf.columns.tolist() == df_prepared.columns.tolist()
assert not clean_op_schijf.isna().any().any()
assert not prepared_op_schijf.isna().any().any()
assert set(prepared_op_schijf['class'].unique()) == {0, 1}

print('Leesbare data:', clean_pad)
print('Numerieke data:', prepared_pad)
print('Aantal voorbereide rijen:', len(prepared_op_schijf))
print('Aantal numerieke invoerkenmerken:', len(prepared_op_schijf.columns) - 1)

Leesbare data: C:\Allerlaatste\DeepLearning\CloudAI\MushroomDataset\prepared\mushroom_clean.csv
Numerieke data: C:\Allerlaatste\DeepLearning\CloudAI\MushroomDataset\prepared\mushroom_prepared.csv
Aantal voorbereide rijen: 60923
Aantal numerieke invoerkenmerken: 159


## 9. Klaar voor de volgende projectstap

De cleaning is reproduceerbaar en het resultaat bevat alle unieke voorbeelden. Onbekende informatie is zichtbaar gebleven, de metingen zijn niet veranderd en de categorische invoer heeft geen verzonnen rangorde.

De ringtegenstrijdigheden en de twee niet-volledig beschreven codes uit de bron blijven bekende beperkingen. De data blijft een simulatie vanuit 173 soorten; dit is geen representatieve veldsteekproef.

Later splitsen we de data passend op in training, validatie en test, zoals in **Models**. Gebruik bij classificatie stratificatie en een vaste `random_state`; neem niet zomaar de bovenste rijen. Houd de testset daarna apart voor de eindbeoordeling. Als de vraag prestaties op volledig nieuwe soorten is, moet ook de gedeelde simulatiebron worden meegenomen in de validatieopzet.

Hier stoppen we. Modellen vergelijken, scaling kiezen en deployment behoren tot de volgende opdrachtfase. Citi Bike wordt afzonderlijk behandeld.